In [ ]:
!pip install pyodbc
!pip install sqlalchemy

In [ ]:
import pyodbc   
from sqlalchemy import create_engine
import pandas as pd

In [ ]:
server_name = 'YOUR_SQL_SERVER_INSTANCE'
database_name = 'YOUR_DATABASE_NAME'


# username = ''
# password = ''


# Connection string:
# Connection_string = (
#                        f"DRIVER = {{ODBC Driver 17 for SQL Server}};"
#                        f"SERVER = {server_name}; DATABASE = {database_name};"
#                         "Trusted_Connection = yes;"
#                     )


# depending upon what rdbms system you are using, the below change.
connection_string = f"mssql+pyodbc://@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server"

In [ ]:
connection_string

In [ ]:
connection2 = create_engine(connection_string)
connection2

In [ ]:
## just to check if the connection is successful or not

query = "SELECT TOP 5 * FROM products_p"

df = pd.read_sql(query, connection2)

df.head()

In [ ]:
#--------------------- Importing the cleaned tables

In [ ]:
query = "SELECT * FROM products_p"

products = pd.read_sql(query, connection2)

products.head()

In [ ]:
products.shape
products.info()
products.isnull().sum()

In [ ]:
query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
"""

pd.read_sql(query, connection2)

In [ ]:
tables = [
    'customers',
    'orders_o',
    'order_items',
    'order_payments_op',
    'order_reviews_or',
    'sellers',
    'geolocations_gl',
    'product_category_name_translation', 'order_payments'
]

dataframes = {}

for i in tables:
    dataframes[i] = pd.read_sql(
        f"SELECT * FROM [{i}]",
        connection2
    )
    print(i, dataframes[i].shape)

In [ ]:
customers = dataframes['customers']
orders = dataframes['orders_o']
order_items = dataframes['order_items']
order_payments = dataframes['order_payments_op']
order_reviews = dataframes['order_reviews_or']
sellers = dataframes['sellers']
geolocation = dataframes['geolocations_gl']
product_category_name_translation = dataframes['product_category_name_translation']
order_payment_raw = dataframes['order_payments']

In [ ]:
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
table = {'customers':customers, 'orders':orders, 'order_items':order_items,'order_payments':order_payments,
         'order_reviews':order_reviews, 'sellers':sellers, 'geolocation':geolocation, 'products':products, 
         'product_category_name_translation':product_category_name_translation}

In [ ]:
# Column-wise missing values and percentage

for i, j in table.items():
    missing = j.isnull().sum()
    missing_pct = (missing / len(j) * 100)

    missing_summary = pd.DataFrame({'missing_values': missing,'missing_percentage': missing_pct})

    missing_summary = missing_summary[missing_summary['missing_values'] > 0]

    if not missing_summary.empty:
        print(f'\n--- {i} ---')
        print(missing_summary)

In [ ]:
print(orders.isna().sum(),'\n',orders.isna().sum()/len(orders)*100)

In [ ]:
print(order_reviews.isna().sum(),'\n',order_reviews.isna().sum()/len(orders)*100)

In [ ]:
print(geolocation.isna().sum(),'\n',geolocation.isna().sum()/len(orders)*100)

In [ ]:
print(products.isna().sum(),'\n',products.isna().sum()/len(orders)*100)

In [ ]:
#---------------------------------------------------------------------------------
#                                  EDA
#---------------------------------------------------------------------------------

In [ ]:
#============================ Data Preparation =============================

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

In [ ]:
## Merging order and order items table, to connect products, order status, customer, purchase date 

In [ ]:
sales = order_items.merge(orders, on = 'order_id', how = 'left', validate='many_to_one')
sales

In [ ]:
sales.nunique()

In [ ]:
print("Order status distribution:")
print(orders['order_status'].value_counts(dropna=False))

In [ ]:
missing_approval = orders[(orders['order_status'] == 'delivered') & (orders['order_approved_at'].isnull())]

print("Delivered orders with missing approval date:",len(missing_approval))

In [ ]:
## Adding a revenue column ---> This is not the profit

In [ ]:
## Total order-item value = product price + freight charegs
# This is not profit 

sales['total_revenue'] = sales['price'] + sales['freight_value']
sales

In [ ]:
#======================================== Business KPIs ===================================

In [ ]:
## Basic KPIs

In [ ]:
total_orders = orders['order_id'].nunique()
total_orders

In [ ]:
orders_with_items = sales['order_id'].nunique()
orders_with_items

In [ ]:
product_sales = sales['price'].sum()
product_sales

In [ ]:
freight_value = sales['freight_value'].sum()
freight_value

In [ ]:
total_sales = sales['total_revenue'].sum()
total_sales

In [ ]:
delivered_sales = sales[sales['order_status'] == 'delivered']
delivered_sales

In [ ]:
# Delivered order revenue

delivered_revenue = delivered_sales['total_revenue'].sum()
delivered_revenue

In [ ]:
delivered_orders = delivered_sales['order_id'].nunique()
delivered_orders

In [ ]:
### Average order value

In [ ]:
all_status_aov = (total_sales / orders_with_items)
all_status_aov

In [ ]:
aov_delivered = delivered_revenue / delivered_orders
aov_delivered

In [ ]:
### cancelled orders and rate of cancellation

In [ ]:
cancelled_orders = (orders['order_status'] == 'canceled').sum()
cancelled_orders

In [ ]:
cancellation_rate = (cancelled_orders / total_orders * 100)
cancellation_rate

# 0.62% cancellation rate

In [ ]:
#=================================== Monthly sales and Revenue analysis ===============================

In [ ]:
sales['year_month'] = sales['order_purchase_timestamp'].dt.to_period('M')
sales.head()

In [ ]:
# Monthly revenue and orders
monthly_sales = sales.groupby('year_month').agg({'total_revenue' : 'sum', 'price':'sum', 'freight_value' : 'sum', 'order_id' : 'nunique'})
monthly_sales

In [ ]:
monthly_sales["MoM_growth_%"] = (monthly_sales["total_revenue"].pct_change() * 100)
monthly_sales

In [ ]:
## Monthly revenue 

plt.figure(figsize=(16,6))

sns.lineplot(monthly_sales,x = monthly_sales.index.astype(str), y = 'total_revenue', marker = 'o', markersize = 8)

plt.title('Monthly Revenue')
plt.xlabel('Months')
plt.ylabel('Revenue')
plt.xticks(rotation = 35)
plt.yticks(range(0,1300000, 100000))
plt.grid()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(11,5))

sns.barplot(monthly_sales,x = monthly_sales.index.astype('str'), y = 'order_id')

plt.title('Monthly Orders')
plt.xlabel('Months')
plt.ylabel('Unique Orders')
plt.xticks(rotation = 35)
plt.grid(axis='y')
plt.tight_layout()

In [ ]:
monthly_sales['revenue_growth_%'] = (monthly_sales['total_revenue'].pct_change() * 100)
monthly_sales['order_growth_%'] = (monthly_sales['order_id'].pct_change() * 100)
monthly_sales

In [ ]:
#========================= Daily revenue and sales analysis ================================

In [ ]:
sales['order_date'] = sales.order_purchase_timestamp.dt.date

daily_sales = sales.groupby('order_date').agg({'total_revenue' : 'sum', 'order_id' : 'nunique'}).reset_index()

daily_sales

In [ ]:
plt.figure(figsize=(14,6))

sns.lineplot(daily_sales, y='total_revenue', x= 'order_date')

plt.title('Daily Revenue Trend')
plt.xlabel('Date')
plt.ylabel('Revenue ')

In [ ]:
#====================================== Weekday sale Analysis ================================

In [ ]:
sales['weekday'] = sales['order_purchase_timestamp'].dt.day_name()
sales

In [ ]:
weekday_sales = sales.groupby('weekday').agg({'total_revenue' : 'sum', 'order_id' : 'nunique'}).reindex(['Monday', 'Tuesday', 'Wednesday','Thursday', 'Friday', 'Saturday', 'Sunday'])
weekday_sales = weekday_sales.reset_index()
weekday_sales

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(weekday_sales, x = 'weekday', y = 'order_id')

plt.title('Weekly Orders')
plt.xlabel('Weekday')
plt.ylabel('Unique Orders')

In [ ]:
#=========================================== Hourly Sales Analysis ================================

In [ ]:
sales['purchase_hour'] = sales['order_purchase_timestamp'].dt.hour

In [ ]:
hourly_sales = sales.groupby('purchase_hour').agg(total_revenue=('total_revenue', 'sum'),total_orders=('order_id', 'nunique')).reset_index()
hourly_sales

In [ ]:
plt.figure(figsize=(12, 5))

sns.lineplot(data=hourly_sales,x='purchase_hour',y='total_orders',marker='o')
plt.title('Orders by Purchase Hour')
plt.xlabel('Hour of Day (0�23)')
plt.ylabel('Unique Orders')
plt.xticks(range(24))
plt.tight_layout()

In [ ]:
#======================================= Product category wise revenue ======================================

In [ ]:
category_sales = sales.merge(products, on = 'product_id', how = 'left')
category_sales

In [ ]:
category_summary = category_sales.groupby('product_category_name_english', dropna = False).agg(
    revenue=('total_revenue', 'sum'), orders=('order_id', 'nunique'), items=('product_id', 'count')).reset_index()
category_summary = category_summary.sort_values('revenue', ascending=False)
category_summary

In [ ]:
top_category = category_summary.head(10)
top_category

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(top_category, y = 'product_category_name_english', x = 'revenue')

plt.title('Top 10 Categories by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Product Category')
plt.xticks(range(0, 1500000, 100000))
plt.tight_layout()

In [ ]:
order_value = sales.groupby('order_id').agg(order_revenue=('total_revenue', 'sum')).reset_index()

print("Average Order Value:", round(order_value['order_revenue'].mean(), 2))
print("Median Order Value:", round(order_value['order_revenue'].median(), 2))
print("Highest Order Value:", round(order_value['order_revenue'].max(), 2))

In [ ]:
order_value

In [ ]:
payment_summary = order_payment_raw.groupby('payment_type').agg(total_payment=('payment_value', 'sum'),payment_records=('order_id', 'count'),
                                                                unique_orders=('order_id', 'nunique')).reset_index()

payment_summary = payment_summary.sort_values('total_payment', ascending=False)

payment_summary = payment_summary[payment_summary['payment_type'] != 'not_defined']
payment_summary

In [ ]:
plt.figure(figsize=(6, 4))

sns.barplot(data=payment_summary,x='payment_type',y='total_payment')

plt.title('Top Payment method')
plt.xlabel('Payment Method')
plt.ylabel('Payment Value')
plt.tight_layout()
plt.show()

In [ ]:
#=================== State- wise revenue =======================

In [ ]:
sales_with_state = delivered_sales.merge(customers[['customer_id', 'customer_state']], how = 'left', on = 'customer_id', validate='many_to_one')
sales_with_state

In [ ]:
revenue_state = sales_with_state.groupby('customer_state').agg({'total_revenue':'sum', 'order_id' : 'nunique'}).sort_values('total_revenue', ascending=False)
revenue_state

In [ ]:
#=============================== payments type by orders ===================================

In [ ]:
orders_by_payment = order_payment_raw.groupby('payment_type').agg(order_count=('order_id', 'nunique')).sort_values('order_count', ascending=False).reset_index()
orders_by_payment = orders_by_payment[orders_by_payment['payment_type'] != 'not_defined']
orders_by_payment

In [ ]:
orders_by_payment

In [ ]:
###============================== Orders by States ==============================

In [ ]:
orders_with_state = orders.merge(customers[['customer_id', 'customer_state']],on='customer_id',how='left',validate='many_to_one')
orders_with_state.columns

In [ ]:
orders_with_state = orders_with_state.groupby('customer_state').order_id.count().sort_values( ascending=False)
orders_with_state

In [ ]:
#=========================== average Installments by payment type ========================================

In [ ]:
avg_installment_by_payment = order_payment_raw.groupby('payment_type').agg(avg_installments=('payment_installments',
                                                                                          'mean')).sort_values('avg_installments', ascending=False)
avg_installment_by_payment

In [ ]:
orders_with_unique_customer = orders.merge(customers[['customer_id', 'customer_unique_id']],on='customer_id',how='left')

first_purchase = orders_with_unique_customer.groupby('customer_unique_id').agg(first_purchase_date=('order_purchase_timestamp', 'min')).reset_index()

first_purchase['acquisition_month'] = first_purchase['first_purchase_date'].dt.to_period('M')

monthly_acquisition = first_purchase.groupby('acquisition_month').size().rename('new_customers').reset_index().sort_values('acquisition_month')

monthly_acquisition

In [ ]:
#======================== New customers revenue month wise ======================================

In [ ]:
new_customer_sales = delivered_sales.merge(customers[['customer_id', 'customer_unique_id']],on='customer_id',how='left')

new_customer_sales['purchase_month'] = new_customer_sales['order_purchase_timestamp'].dt.to_period('M')

new_customer_sales = new_customer_sales.merge(first_purchase[['customer_unique_id', 'acquisition_month']],on='customer_unique_id',how='left')

new_customer_sales = new_customer_sales[new_customer_sales['purchase_month'] == new_customer_sales['acquisition_month']]

monthly_new_customer_revenue = new_customer_sales.groupby('acquisition_month').agg(new_customer_revenue=('total_revenue', 'sum'),new_customers=('customer_unique_id', 'nunique')).reset_index().sort_values('acquisition_month')

monthly_new_customer_revenue

In [ ]:
customer_order_volume = orders.merge(customers[['customer_id', 'customer_unique_id']],on='customer_id',how='left')

customer_order_volume = customer_order_volume[customer_order_volume['order_status'] == 'delivered'].groupby('customer_unique_id').agg(order_count=('order_id',
                                                                                                                                                   'nunique')).reset_index()

customer_order_volume['order_volume_segment'] = pd.cut(customer_order_volume['order_count'],bins=[0, 1, 3, float('inf')],labels=['Low',
                                                                                                                                 'Medium', 'High'])

volume_summary = customer_order_volume.groupby('order_volume_segment', observed=False).agg(customers=('customer_unique_id',
                                                                                    'nunique'),average_orders=('order_count', 'mean')).reset_index()

volume_summary

In [ ]:
#====================== delivery Performance analysis ============================

In [ ]:
# ==== Delivery time and Late delivery ====

In [ ]:
sales['delivery_time_days'] = (sales['order_delivered_customer_date'] -sales['order_purchase_timestamp']).dt.days

sales['late_delivery'] = sales['order_delivered_customer_date'] >sales['order_estimated_delivery_date']

sales[['order_id', 'delivery_time_days', 'late_delivery']].head()

In [ ]:
delivered = sales[sales['order_status'] == 'delivered'].copy()

delivery_summary = delivered.agg(avg_delivery_days=('delivery_time_days', 'mean'),median_delivery_days=('delivery_time_days', 'median'))

late_delivery_rate = (delivered.drop_duplicates('order_id')['late_delivery'].mean() * 100)

print(delivery_summary)
print(f"Late delivery rate: {late_delivery_rate:.2f}%")

In [ ]:
#======================= Customer Review analysis ============================

In [ ]:
print("Review Score Distribution:")
print(order_reviews['review_score'].value_counts().sort_index())

print("\nAverage Review Score:")
print(order_reviews['review_score'].mean().round(2))

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(data=order_reviews, x='review_score')

plt.title('Customer Review Score Distribution')
plt.xlabel('Review Score')
plt.ylabel('Number of Reviews')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
review_summary = order_reviews.assign(review_category=np.where(order_reviews['review_score'] >= 4,'Positive',
                    np.where(order_reviews['review_score'] <= 2, 'Negative', 'Neutral')))['review_category'].value_counts()

review_summary

In [ ]:
plt.pie(review_summary,autopct='%1.1f%%', labels= review_summary.index)
plt.title('Review Category')

In [ ]:
## ============= Reveiw score vs delivery delay ===============

In [ ]:
review_delivery = (orders[['order_id', 'order_status','order_delivered_customer_date','order_estimated_delivery_date']].merge(
                    order_reviews[['order_id', 'review_score']],on='order_id',how='inner', validate='one_to_many'))

review_delivery

In [ ]:
review_delivery = review_delivery[review_delivery['order_status'] == 'delivered'].copy()

review_delivery['delivery_status'] = np.where(review_delivery['order_delivered_customer_date'] >review_delivery['order_estimated_delivery_date'],
                                              'Late','On Time')
review_delivery

In [ ]:
delivery_review_summary = review_delivery.groupby('delivery_status').agg(total_reviews=('review_score', 'count'),
                                                                          avg_review_score=('review_score', 'mean')).reset_index()

delivery_review_summary

In [ ]:
sns.barplot(data=delivery_review_summary,x='delivery_status',y='avg_review_score')

plt.title('Delivery Status vs Average Review Score')
plt.xlabel('Delivery Status')
plt.ylabel('Average Review Score')
plt.yticks(range(0,6,1))
plt.show()

In [ ]:
####====================================== Customer repeat purchase analysis =====================================

In [ ]:
## Purchase frequency & the repeat %age of the customers

customer_orders = orders[orders['order_status'] == 'delivered'][['order_id', 'customer_id']].merge(customers[['customer_id','customer_unique_id']],on='customer_id',how='left')

customer_orders = customer_orders.groupby('customer_unique_id')['order_id'].nunique().reset_index(name='total_orders')

customer_orders['purchase_type'] = np.where(customer_orders['total_orders'] > 1,'Repeat Customer','One-time Customer')

customer_orders['purchase_type'].value_counts().rename_axis('customer_type')

purchase_summary['percentage'] = (purchase_summary['customers']/ purchase_summary['customers'].sum() * 100).round(2)

purchase_summary

In [ ]:
plt.pie(data=purchase_summary, x = 'percentage', autopct='%1.2f%%',labels='customer_type', startangle=90)

plt.title('One-time vs Repeat Customers')
plt.xlabel('')
plt.ylabel('Customers (%)')
plt.tight_layout()
plt.show()

In [ ]:
# =================================================== Seller Performance analysis =====================================================

In [ ]:
sales['seller_short_id'] = sales['seller_id'].map({v: f'S_{i+1}' for i, v in enumerate(sales['seller_id'].dropna().unique())})
sales

In [ ]:
# ------------ Seller Wise Performance ----------------

seller_performance = sales[sales['order_status'] == 'delivered'].groupby('seller_id').agg(total_orders=('order_id',
                                'nunique'),total_revenue=('total_revenue', 'sum')).sort_values('total_revenue', ascending=False).reset_index()

seller_performance

In [ ]:
#---------------- Top 10 Sellers by Revenue

plt.figure(figsize=(10, 5))

sns.barplot(data=seller_performance.head(10),x='total_revenue',y='seller_id')

plt.title('Top 10 Sellers by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Seller ID')
plt.tight_layout()
plt.show()

In [ ]:
#====================================== Category- Wise customer satisfaction ========================================

In [ ]:
category_reviews = order_items.merge(products[['product_id',
                                    'product_category_name_english']], on='product_id').merge(order_reviews[['order_id',
                                                                'review_score']], on='order_id')
category_reviews

In [ ]:
category_satisfaction = category_reviews.groupby('product_category_name_english').agg(avg_review_score=('review_score',
                                    'mean'),total_reviews=('review_score', 'count'))

category_satisfaction = category_satisfaction[category_satisfaction['total_reviews'] >= 100].sort_values('avg_review_score'
                                                                                                        , ascending = False).reset_index()

category_satisfaction.head(10)

In [ ]:
category_satisfaction.tail(10).sort_values('avg_review_score')

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(data=category_satisfaction.sort_values('avg_review_score').head(10),x='avg_review_score',y='product_category_name_english')

plt.title('10 Lowest-Rated Product Categories')
plt.xlabel('Average Review Score')
plt.ylabel('Product Category')
plt.xticks(range(0,6))
plt.show()

In [ ]:
#============================== Freight cost vs Product price analysis ======================================

In [ ]:
# Freight percentage 


sales['freight_percentage'] = (sales['freight_value'] / sales['price'] * 100)

freight_cpt  = sales[['price', 'freight_value', 'freight_percentage']].describe()
freight_cpt

In [ ]:
freight_cpt['freight_percentage']

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=sales,x='price',y='freight_value')


plt.title('Freight Cost vs Product Price')
plt.xlabel('Product Price')
plt.ylabel('Freight Value')
plt.show()

In [ ]:
# ============================= outlier analysis ================================

In [ ]:

cols = ['price', 'freight_value', 'total_revenue']

for i in cols:
    Q1 = sales[i].quantile(0.25)
    Q3 = sales[i].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((sales[i] < lower) | (sales[i] > upper)).sum()

    print(f'{i}: {outliers} outliers')

In [ ]:
for i in cols:
    Q1 = sales[i].quantile(0.25)
    Q3 = sales[i].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR
    count = (sales[i] > upper).sum()

    print(f'{i}: {count} upper outliers ({count / len(sales) * 100:.2f}%)')

sales[['price', 'freight_value', 'total_revenue']].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]).round(2)

In [ ]:

plt.figure(figsize=(10, 5))
sns.boxplot(data=sales[cols])
plt.title('Outlier Detection in Sales Data')
plt.ylabel('Value')
plt.tight_layout()
plt.show()


In [ ]:
## ========================== Correlation Analysis ==============================

In [ ]:
sns.heatmap(sales[cols].corr(),annot=True,fmt='.2f')

plt.title('Correlation Between Price, Freight and Revenue')


# Key Business Insights and Recommendations

## 1. Sales Performance
Monthly and daily trends were analysed to identify sales
patterns and periods of higher activity.

**Recommendation:** Use observed demand patterns to support
inventory and campaign planning.

## 2. Product Categories
Category-level revenue and review scores were analysed.

**Recommendation:** Investigate low-rated categories to identify
potential product, delivery or service issues.

## 3. Regional Performance
State-level revenue and order volumes were compared.

**Recommendation:** Review regional demand and logistics needs
before allocating additional resources.

## 4. Delivery and Customer Satisfaction
Delivery timeliness was compared with customer review scores.

**Recommendation:** Investigate delayed deliveries and prioritise
improvements in delivery reliability.

## 5. Customer Retention
One-time and repeat customers were identified using
customer_unique_id.

**Recommendation:** Evaluate retention initiatives using
repeat purchase behaviour.

## 6. Seller Performance
Seller order-item value was compared across all statuses
and delivered orders.

**Recommendation:** Assess seller performance alongside order
volume and delivery outcomes.

## 7. Shipping Costs
Freight value was compared with product price.

**Recommendation:** Investigate categories or regions where
shipping costs are high relative to product prices.

## 8. Data Limitations
Outliers were identified but not automatically removed.
Missing values and order-level review associations were
considered when interpreting the results.
